# Pharma A/B Test: Packaging Impact in Mobile App

## Check if Blue packaging (B) is better than Red (A)

In [3]:
import pandas as pd
df=pd.read_csv("pharma_ab_test_data.csv")
df.head()

,user_id,group,pack_color,age_group,gender,device_type,visit_date,visit_time,purchase_datetime,time_on_page_sec,scrolled_to_reviews,added_to_cart,converted,previous_app_user,previous_product_buyer
0,1,A,Red,45–54,M,Android,2024-11-11,07:20,NaN,65,1,1,0,1,0
1,2,B,Blue,45–54,F,iOS,2024-11-02,20:32,NaN,65,1,0,0,1,0
2,3,B,Blue,45–54,F,iOS,2024-11-15,14:50,NaN,72,0,1,0,1,0
3,4,A,Red,45–54,F,iOS,2024-11-26,08:25,2024-11-26 16:11,69,1,1,1,1,0
4,5,A,Red,18–24,F,iOS,2024-11-08,03:01,NaN,42,0,0,0,0,0


### The dataset contains user behavior data for two groups:
Group A: Red packaging
Group B: Blue packaging

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   user_id                 1000 non-null   int64 
 1   group                   1000 non-null   object
 2   pack_color              1000 non-null   object
 3   age_group               1000 non-null   object
 4   gender                  1000 non-null   object
 5   device_type             1000 non-null   object
 6   visit_date              1000 non-null   object
 7   visit_time              1000 non-null   object
 8   purchase_datetime       206 non-null    object
 9   time_on_page_sec        1000 non-null   int64 
 10  scrolled_to_reviews     1000 non-null   int64 
 11  added_to_cart           1000 non-null   int64 
 12  converted               1000 non-null   int64 
 13  previous_app_user       1000 non-null   int64 
 14  previous_product_buyer  1000 non-null   int64 
dtypes: in

### Insight:

- The dataset contains 1000 user records.
- Most columns have no missing values.
- The `purchase_datetime` column has missing values, which is expected because not all users made a purchase.
- The key column for analysis is `converted` (1 = purchase, 0 = no purchase).
- The dataset looks clean and ready for analysis.

In [5]:
df['group'].value_counts()

group
A    511
B    489
Name: count, dtype: int64

### Insight:
- A ≈ B users
- Split is balanced
- No issue, continue

## Conversion rate = (number of purchases / total users)

In [6]:
df.groupby('group')['converted'].mean()

group
A    0.160470
B    0.253579
Name: converted, dtype: float64

### Insight:
- A conversion ≈ 16%
- B conversion ≈ 25%
- B is higher → looks better

# Statistical Test

In [8]:
from statsmodels.stats.proportion import proportions_ztest

In [11]:
conversions=df.groupby('group')['converted'].sum()
users=df.groupby('group')['converted'].count()
z_stat,p_value=proportions_ztest(conversions,users)
print("Z-test:",z_stat)
print("p-value:",p_value)

Z-test: -3.6392591510784973
p-value: 0.0002734235320508854


### Insight:
- p-value ≈ 0.00027
- p < 0.05 → result is significant
- Difference is real, not random

## Final Conclusion

- Group B has higher conversion rate than Group A.
- The difference is statistically significant (p < 0.05).
- Therefore, Blue packaging performs better.

Decision: Ship Version B

# FUNNEL ANALYSIS

In [12]:
df.groupby('group')[['scrolled_to_reviews','added_to_cart','converted']].mean()

,scrolled_to_reviews,added_to_cart,converted
group,,,
A,0.362035,0.393346,0.160470
B,0.382413,0.462168,0.253579


### Insight:
- B has higher add-to-cart rate
- B has higher conversion
- B slightly improves scrolling
- B improves full funnel

# SEGMENTATION

In [13]:
df.groupby(['age_group','group'])['converted'].mean().unstack()

group,A,B
age_group,,
18–24,0.164948,0.230769
25–34,0.153061,0.330000
35–44,0.118812,0.193548
45–54,0.188679,0.188889
55+,0.174312,0.313725


### Insight:
- B works better for most age groups
- Strong improvement in 25–34 and 55+
- No difference in 45–54

# Confidence Interval

In [14]:
import statsmodels.api as sm
conv = df.groupby('group')['converted'].sum()
n = df.groupby('group')['converted'].count()
ci_A = sm.stats.proportion_confint(conv['A'], n['A'])
ci_B = sm.stats.proportion_confint(conv['B'], n['B'])
print("A CI:", ci_A)
print("B CI:", ci_B)

A CI: (0.12864584704451387, 0.19229348759345088)
B CI: (0.215018283556066, 0.29213918065661293)


### Insight:
- A CI ≈ 13% to 19%
- B CI ≈ 21% to 29%
- B range is higher → clearly better

# Device Analysis

In [15]:
df.groupby(['device_type','group'])['converted'].mean().unstack()

group,A,B
device_type,,
Android,0.137405,0.244813
iOS,0.184739,0.262097


### Insight:
- B is better on both devices
- Higher improvement on Android
- iOS already higher but still improves

#  User Type Analysis

In [16]:
df.groupby(['previous_app_user','group'])['converted'].mean().unstack()

group,A,B
previous_app_user,,
0,0.176768,0.253886
1,0.150160,0.253378


### Insight:
- B is better for both new and old users
- Strong improvement for returning users
- B works consistently across user types